# HOD fit results

Results of a fit made by `jobs/hod_fit.sh` (`scripts/run_hod_fit.py`), read from its folder
`hicola_data/hicola/<target>/<model>/<realization>/HiCOLA_<sim>_z<z>_HOD_fit_<fit_name>/`.
The fit name says what was fitted: `<tracer>_<HOD model>_<n free>p_<ptcl|nfw>[_<statistics>]`
(satellites on member particles or on the NFW profile; the statistics only when not all of
wp + $\xi_0$ + $\xi_2$), e.g. `LRG_SHOD_6p_ptcl`, `LRG_SHOD_6p_nfw_wp`. Files:

| file | content |
|---|---|
| `settings.json` | settings of the fit (data, priors, minimizer, seeds), run, code versions |
| `<HOD parameter file>.yaml` | the HOD model and the fixed parameters |
| `data.npz` | the fitted data: separations, values, errors, data vector, covariance |
| `evaluations/rank_<k>.txt` | every evaluation (parameters, $\chi^2$, galaxies), written as the fit runs |
| `iterations.txt` | best point of each iteration of the minimizer |
| `best_fit.json` | best-fit parameters, $\chi^2$, minimizer status (at the end) |
| `best_fit_model.npz` | model at the best fit (fit seed + extra seeds), HOD occupation (at the end) |
| `reference_model.npz` | the same for the HOD of the reference parameter file (`REFERENCE_PARAM_FILE`), if any |

While the fit runs (or if it stopped before the end) the notebook shows the best evaluation so far.

In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
target, model, realization = 'Abacus_Small', 'nDGP_N1.2', 's1234'
redshift = 0.5
fit_name = 'LRG_SHOD_6p_ptcl_r1-max'   # named after what is fitted; printed by jobs/hod_fit.sh (line 'Fit:')

In [ ]:
fit = an.load_hod_fit(target, model, realization, redshift, fit_name)
settings, data, evaluations, best_fit, fit_model = (fit[key] for key in ('settings', 'data', 'evaluations', 'best_fit', 'model'))
ref_model = fit['reference']                         # model of the reference HOD (None without one)
reference = best_fit.get('reference') if best_fit else None
names = list(settings['priors'])                      # free parameters
columns = [f'{name}_{settings["tracer"]}' for name in names]   # their columns in the evaluation files
finished = best_fit is not None

print(fit['dir'])
if finished:
    print(f"Finished {best_fit['finished']}: {best_fit['minimizer']['message']}, {best_fit['minimizer']['nit']} iterations, "
          f"{best_fit['minimizer']['nfev']} evaluations in {best_fit['time_h']:.2f} h")
else:
    n_eval = 0 if evaluations is None else evaluations.size
    print(f'Not finished (running, or stopped before the end): {n_eval} evaluations so far, '
          f'started {settings["started"]}. The results below use the best evaluation so far.')

## Settings

In [ ]:
d, m, mz = settings['data'], settings['minimizer'], settings['model']
rows = {
    'run': f"{settings['run']}, z = {settings['redshift']}, box {settings['boxsize']} Mpc/h",
    'halo catalog': settings['halo_catalog'],
    'satellites': settings.get('satellites', 'particles' if settings.get('particles') else 'nfw'),
    'HOD': f"{settings['tracer']}, {settings['hod_param_file']}",
    'free parameters': ', '.join(f'{n} [{lo}, {hi}]' for n, (lo, hi) in settings['priors'].items()),
    'statistics': f"{' + '.join(d['statistics'])} ({d['n_data']} values)",
    'fit range': ('all bins' if d.get('fit_r_range', [None, None]) == [None, None]
                  else f"{d['fit_r_range'][0]} to {d['fit_r_range'][1]} Mpc/h (rp and s; None = no limit)"),
    'covariance': f"jackknife errors x {Path(d['correlation']).name} (covering "
                  f"{' + '.join(d.get('correlation_statistics', d['statistics']))}; blocks of the fitted statistics)",
    'model': f"RSD along {mz['los']}, H(z)/H0 = {float(settings['cosmology']['efunc']):.5f}, mock seed {mz['mock_seed']}, extra seeds {mz['model_seeds']}",
    'minimizer': f"{m['package']} {m['method']}, maxiter {m['maxiter']}, popsize {m['popsize']}, xtol {m['xtol']}",
    'start': f"{m['start']} (seed {m['start_seed']})",
    'resources': f"{settings['resources']['mpi_ranks']} MPI ranks x {settings['resources']['threads_per_rank']} threads",
    'Hi-COLA': settings['code']['hicola'],
    'HODDIES': settings['code']['hoddies'],
    'job log': settings['job_log'],
}
if settings.get('particles'):
    pt = settings['particles']
    rows['member particles'] = f"{Path(pt['file']).name}: {pt['n']:,} (fraction {pt['fraction']}, seed {pt['seed']})"
if 'wp' in d['statistics']:
    rows['wp'] = f"rp in {d['rp_range']} Mpc/h, rebin {d['rp_rebin']}, pi_max {d['pimax']}; {Path(d['rppi']).name}"
if 'xi_ells' in d['statistics']:
    rows['xi_ell'] = f"ells {d['ells']}, s in {d['s_range']} Mpc/h, rebin {d['s_rebin']}; {Path(d['smu']).name}"
display(pd.DataFrame({'setting': list(rows.values())}, index=list(rows)).style.set_properties(**{'text-align': 'left'}))

## Best fit

In [ ]:
if finished:
    best = np.array([best_fit['parameters'][c] for c in columns])
    best_chi2 = best_fit['chi2']
else:
    row = evaluations[np.argmin(evaluations['chi2'])]
    best = np.array([row[c] for c in columns])
    best_chi2 = float(row['chi2'])

low, high = np.array([settings['priors'][n] for n in names]).T
position = (best - low) / (high - low)
table = pd.DataFrame({
    'prior min': low, 'prior max': high,
    'start': [m['x0'][c] for c in columns],
    'best fit': best,
    **({'reference': [reference['parameters'][c] for c in columns]} if reference else {}),
    'position in prior': position,
    'at a prior edge': np.where((position < 0.02) | (position > 0.98), 'yes', ''),
}, index=names)
display(table.style.format({'prior min': '{:g}', 'prior max': '{:g}', 'start': '{:.4f}', 'best fit': '{:.4f}', 'reference': '{:.4f}',
                            'position in prior': '{:.0%}'}))

In [ ]:
# chi2 of each block of the data vector (with its own block of the covariance)
blocks = []
if 'wp' in d['statistics']:
    blocks.append(('wp', 'rp', data['wp'], data['wp_err']))
if 'xi_ells' in d['statistics']:
    for i, ell in enumerate(data['ells']):
        blocks.append((f'xi{ell}', 's', data['xi'][i], data['xi_err'][i]))
offsets = np.cumsum([0] + [b[2].size for b in blocks])
block_index = {name: np.arange(offsets[i], offsets[i + 1]) for i, (name, *_) in enumerate(blocks)}

summary = {'chi2': best_chi2}
if finished:
    model_vec = fit_model['model_vec'][0]
    residual = data['data_vec'] - model_vec
    for name, index in block_index.items():
        sub = np.linalg.inv(data['cov'][np.ix_(index, index)])
        summary[f'chi2 {name} ({index.size} bins)'] = residual[index] @ sub @ residual[index]
    if reference:
        ref_residual = data['data_vec'] - ref_model['model_vec'][0]
        summary['reference HOD: chi2'] = reference['chi2']
        for name, index in block_index.items():
            sub = np.linalg.inv(data['cov'][np.ix_(index, index)])
            summary[f'reference HOD: chi2 {name}'] = ref_residual[index] @ sub @ ref_residual[index]
        summary['reference HOD: number density [(h/Mpc)^3]'] = f"{reference['number_density']:.3e}"
        summary['reference HOD: satellite fraction'] = reference['satellite_fraction']
        if reference['other_differences']:
            summary['reference HOD: other values differing from the fit'] = str(reference['other_differences'])
    summary.update({
        'n_data': best_fit['n_data'], 'n_free': best_fit['n_free'], 'dof': best_fit['dof'],
        'chi2 / dof': best_fit['chi2_per_dof'], 'p-value': best_fit['p_value'],
        'chi2 with the extra seeds (mean +- std)': f"{fit_model['chi2'][1:].mean():.2f} +- {fit_model['chi2'][1:].std(ddof=1):.2f}"
                                                   if fit_model['chi2'].size > 2 else '',
        'galaxies': f"{best_fit['n_galaxies']:,}", 'number density [(h/Mpc)^3]': f"{best_fit['number_density']:.3e}",
        'satellite fraction': best_fit['satellite_fraction'],
        **({'satellites on the NFW profile (too few stored particles)': best_fit['sat_nfw_fraction']}
           if best_fit.get('sat_nfw_fraction') is not None else {}),
        'iterations / evaluations': f"{best_fit['minimizer']['nit']} / {best_fit['minimizer']['nfev']}",
        'failed evaluations': best_fit['minimizer']['failed_evaluations'],
        'minimizer status': best_fit['minimizer']['message'], 'time [h]': best_fit['time_h'],
    })
display(pd.DataFrame({'value': [f'{v:.4g}' if isinstance(v, float) else v for v in summary.values()]}, index=list(summary)))

## Data and best-fit model

Data with their errors (square root of the diagonal of the covariance); model at the best fit with
the fit's mock seed (line) and the mean $\pm$ standard deviation over the extra seeds (band);
dashed: the HOD of the reference parameter file (`REFERENCE_PARAM_FILE` of the job, same seeds);
residuals in units of the errors.

In [ ]:
if not finished:
    print('The best-fit model is computed at the end of the fit.')
else:
    vectors = fit_model['model_vec']
    fig, axes = plt.subplots(2, len(blocks), figsize=(3.6 * len(blocks), 4.2), dpi=150, sharex='col',
                             gridspec_kw={'height_ratios': (3, 1.2), 'hspace': 0.05}, squeeze=False, constrained_layout=True)
    for j, (name, sep_key, values, errors) in enumerate(blocks):
        x = data[sep_key]
        index = block_index[name]
        weight = x if name == 'wp' else x**2      # rp wp and s^2 xi
        ax, rax = axes[0, j], axes[1, j]
        ax.errorbar(x, weight * values, weight * errors, fmt='o', ms=2.5, color='0.15', lw=0.8, label='data')
        ax.plot(x, weight * vectors[0, index], color='#c47f2b', lw=1.4, label=f'best fit (seed {fit_model["seeds"][0]})')
        if vectors.shape[0] > 2:
            mean, std = vectors[1:, index].mean(0), vectors[1:, index].std(0, ddof=1)
            ax.fill_between(x, weight * (mean - std), weight * (mean + std), color='#c47f2b', alpha=0.25, lw=0,
                            label=f'{vectors.shape[0] - 1} other seeds')
        if ref_model is not None:
            ref_vectors = ref_model['model_vec']
            ax.plot(x, weight * ref_vectors[0, index], color='#1f4e79', lw=1.2, ls='--',
                    label=f'reference HOD ({Path(reference["param_file"]).name})')
        rax.axhline(0, color='0.4', lw=0.8, ls=':')
        rax.plot(x, (values - vectors[0, index]) / errors, 'o-', ms=2.5, lw=0.8, color='#c47f2b')
        if ref_model is not None:
            rax.plot(x, (values - ref_vectors[0, index]) / errors, 's--', ms=2, lw=0.8, color='#1f4e79')
        ax.set_xscale('log')
        ax.set_ylabel(r'$r_p\,w_p$ [$(\mathrm{Mpc}/h)^2$]' if name == 'wp' else rf'$s^2\xi_{{{name[2:]}}}$ [$(\mathrm{{Mpc}}/h)^2$]')
        rax.set_xlabel(r'$r_p$ [Mpc/$h$]' if name == 'wp' else r'$s$ [Mpc/$h$]')
        rax.set_ylabel(r'$\Delta/\sigma$')
        ax.set_title(f'{name}, {settings["run"]}, z = {settings["redshift"]}', fontsize=8)
        for a in (ax, rax):
            a.grid(True, which='major', ls=':', color='0.82', lw=0.45)
            a.tick_params(which='both', direction='in', top=True, right=True)
    axes[0, 0].legend(fontsize=7, frameon=False)
    plt.show()

## Convergence

Every evaluation ($\chi^2$ of the mock at that point, all ranks), the best so far, and the best of
each iteration; then the parameters of every evaluation coloured by $\chi^2$, with the prior range
(grey) and the best fit (line).

In [ ]:
if evaluations is None:
    print('No evaluation yet.')
else:
    ok = np.isfinite(evaluations['chi2'])
    order = np.arange(evaluations.size)
    fig, ax = plt.subplots(figsize=(7, 3.2), dpi=150, constrained_layout=True)
    ax.scatter(order[ok], evaluations['chi2'][ok], s=3, color='0.6', lw=0, label='evaluations')
    ax.plot(order[ok], np.minimum.accumulate(evaluations['chi2'][ok]), color='#c47f2b', lw=1.4, label='best so far')
    if fit['iterations'] is not None:
        it = fit['iterations']
        ax.plot(it['n_evaluations'], it['chi2'], 'o', ms=3, color='#1f4e79', label='best of each iteration')
    ax.set_yscale('log')
    ax.set_xlabel('evaluation')
    ax.set_ylabel(r'$\chi^2$')
    ax.legend(fontsize=7, frameon=False)
    ax.grid(True, which='major', ls=':', color='0.82', lw=0.45)
    plt.show()
    print(f'{(~ok).sum()} evaluations without a chi2 (outside the priors or failed: {int(evaluations["failed"].sum())} failed)')

In [ ]:
if evaluations is not None:
    ok = np.isfinite(evaluations['chi2'])
    ncol = 3
    nrow = int(np.ceil(len(names) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.4 * nrow), dpi=150, constrained_layout=True, squeeze=False)
    for ax, name, column, lo, hi, b in zip(axes.flat, names, columns, low, high, best):
        sc = ax.scatter(np.arange(evaluations.size)[ok], evaluations[column][ok], c=np.log10(evaluations['chi2'][ok]),
                        s=3, lw=0, cmap='viridis_r')
        ax.axhspan(lo, hi, color='0.92', zorder=0)
        ax.axhline(b, color='#c47f2b', lw=1.2)
        ax.set_title(name, fontsize=9)
        ax.set_xlabel('evaluation', fontsize=8)
    for ax in axes.flat[len(names):]:
        ax.axis('off')
    fig.colorbar(sc, ax=axes, label=r'$\log_{10}\chi^2$', shrink=0.8)
    plt.show()

## HOD at the best fit

In [ ]:
if finished:
    logm, n_cen, n_sat = fit_model['log_mass'], fit_model['n_cen_hod'], fit_model['n_sat_hod']
    fig, ax = plt.subplots(figsize=(5, 3.4), dpi=150, constrained_layout=True)
    curves = [('best fit', fit_model, '#c47f2b')]
    if ref_model is not None:
        curves.append(('reference', ref_model, '#1f4e79'))
    for label, occupation, color in curves:
        ax.plot(logm, occupation['n_cen_hod'] + occupation['n_sat_hod'], color=color, lw=1.4, label=f'{label}: total')
        ax.plot(logm, occupation['n_cen_hod'], color=color, lw=1.0, ls='--', label=f'{label}: centrals')
        ax.plot(logm, occupation['n_sat_hod'], color=color, lw=1.0, ls=':', label=f'{label}: satellites')
    ax.set_yscale('log')
    ax.set_ylim(1e-3, 20)
    ax.set_xlabel(r'$\log_{10}(M_h\,[M_\odot/h])$')
    ax.set_ylabel(r'$\langle N_\mathrm{gal}\rangle$')
    ax.set_title(f"{settings['tracer']}, {settings['run']}, z = {settings['redshift']}", fontsize=8)
    ax.legend(fontsize=7, frameon=False)
    ax.grid(True, which='major', ls=':', color='0.82', lw=0.45)
    plt.show()